**AI Research Paper Intelligence System**

Data collection

text extraction

NLP preprocessing and cleaning

embedding

transformer

Facebook AI Similarity Search(Faiss) vector database

Semantic search

Summarization using Bi diectional and Auto-Regressive Model(BART)

Implementing Key BERT

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

Made Dataframe

In [ ]:
df=pd.DataFrame(dataset)
df

In [ ]:
df.columns

Drop title and abstract

In [ ]:
df=df[['title','abstract']]

In [ ]:
df

In [ ]:
df.shape

Decrease data

In [ ]:
df=df.head(15000)

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

new column

In [ ]:
df["paper_text"]=df["title"]+" "+df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df[["paper_text"]])

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
from sentence_transformers import SentenceTransformer

used transformer to extract info

In [ ]:
model=SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
sample_text=df["paper_text"].iloc[0]

In [ ]:
sample_text

replaced  \n and space

In [ ]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)

Generate Full Embedding of all Papers

In [ ]:
import numpy as np
import os
if(os.path.exists("embedding.npy")):
  print("File Already Exists")
  embedding=np.load("embedding.npy")
else:
  print("Generating Embedding")
  embedding=model.encode(df["paper_text"].to_list(),batch_size=32,show_progress_bar=True)


In [ ]:
import numpy as np
np.save("embedding.npy",embedding)

In [ ]:
print(embedding.shape)
print(type(embedding))

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
faiss.normalize_L2(embedding)

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding)

In [ ]:
print(index.ntotal)

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D) #Score / Similarity score
print(I) #Indices

In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
print(df.iloc[11873]["title"])

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()

In [ ]:
search_paper("deep learning for medical image analysis")

Summarization

In [ ]:
!pip install transformers==4.46.3
from transformers import pipeline
summarizer=pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
)

In [ ]:
type(summarizer)

In [ ]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
summary[0]["summary_text"]

In [ ]:
for score,idx in zip(D[0],I[0]):
  print("Similarity score",score)
  print("Title",df.iloc[idx]["title"])
  print("Abstract",df.iloc[idx]["abstract"][:500])
  print()
  summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
  print(summary)
  print(summary[0]["summary_text"])
  print()

In [ ]:
def search_and_summarize(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score, idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]["summary_text"])
    print()

In [ ]:
search_and_summarize("Deep learning in medical imaging",5)

KEY BERT implementation

In [ ]:
!pip install keybert==0.8

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model=KeyBERT()

In [ ]:
type(kw_model) #SentenceTransformer

In [ ]:
text=df.iloc[10466]["abstract"]
keywords=kw_model.extract_keywords(text)

In [ ]:
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

In [ ]:
kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")

In [ ]:
print(keywords)

In [ ]:
def search_and_summarize(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score, idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]["summary_text"])
    print()
    keywords=kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")
    print(keywords)
    for k,s in keywords:
      print(k)